In [1]:
library(dplyr)
library(tidyverse)
library(data.table)
library(ggplot2)
library(RColorBrewer)

library(Seurat)

options(stringsAsFactors=F)
options(scipen=100)　


次のパッケージを付け加えます: ‘dplyr’


以下のオブジェクトは ‘package:stats’ からマスクされています:

    filter, lag


以下のオブジェクトは ‘package:base’ からマスクされています:

    intersect, setdiff, setequal, union


── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ forcats   1.0.0     ✔ readr     2.1.5
✔ ggplot2   3.5.1     ✔ stringr   1.5.1
✔ lubridate 1.9.3     ✔ tibble    3.2.1
✔ purrr     1.0.2     ✔ tidyr     1.3.1
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

次のパッケージを付け加えます: ‘data.table’


以下のオブジェクトは ‘package:lubridate’ からマスクされています:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


以下のオブジェクトは ‘package:purrr’ からマスクされています:

    transpose


以下のオブジェクトは ‘package:dplyr’ からマスクされています:

    between, first, last


要求されたパッケージ SeuratObject をロード中です

要求されたパッケージ

In [2]:
data_path=<path_to_data>
project_id.data <- Read10X(data.dir = data_path) 

min_cell=ncol(project_id.data)/100
Seurat_ob <- CreateSeuratObject(counts = project_id.data, project = "project_id", min.cells = min_cell)

In [ ]:
filtered_count_df=as.data.frame(Seurat_ob[["RNA"]]$counts)

In [ ]:
out_f=<output_path>
dir.create(out_f, recursive=TRUE)

In [ ]:
write.table(data.frame(gene=rownames(filtered_count_df), filtered_count_df), paste0(out_f, "/filtered_count.txt") ,row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)


In [ ]:
#option: convert SYMBOL to ENSG


annotation_df=read.table(paste0(data_path, "/features.tsv.gz")
head(annotation_df)
colnames(annotation_df)=c("ENSG", "gene", "GENE", "Expression")

In [ ]:
count_df=read.table(paste0(data_path, "/filtered_count.txt"), header=1)

In [ ]:
#option: convert SYMBOL to ENSG, filter to unique gene id
count_df2=left_join(count_df, annotation_df, by="gene")
unique_df=count_df2[!(duplicated(count_df2$ENSG) | duplicated(count_df2$ENSG, fromLast = TRUE)),] #duplicated関数は重複している初発は含めないので、fromLast=TRUEとの論理和をとる
rownames(unique_df)=unique_df$ENSG
unique_d2=unique_df %>% dplyr::select(-c("gene", "ENSG", "GENE", "Expression"))

In [ ]:
logCPM  = apply(unique_d2, 2, function(x) {log(((x/sum(x)) * 10000 )+1)}) #low expression filter後にlogCPM

## Check
out_f=<output_path>
dir.create(out_f, recursive=T)
write.table(data.frame(gene=rownames(unique_d2), unique_d2), paste0(out_f, "/low_gene_omitted_count.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)

# step2 obtain the intersect genes between TF-GRN matrix and scRNA-seq 

In [ ]:
TF_df=read.table("~/TF_matrix/TF_matrix_tuning/TF_gene_exp_d0_10K/TF_gene_flag_matrix.txt", header=1,row.names=1)
#You can download TF_gene_flag_matrix.txt from data folder.
TF_df2=TF_df %>% separate(gene_name, c("gene", "ver"), sep="\\.") %>% dplyr::select(-c("ver"))
unique_df=TF_df2[!(duplicated(TF_df2$gene) | duplicated(TF_df2$gene, fromLast = TRUE)),]
rownames(unique_df)=unique_df$gene
unique_df2=unique_df %>% dplyr::select(-c("gene"))

In [ ]:

low_exp_omit_path=paste0(out_f, "/low_gene_omitted_count.txt")

out_f3=paste0(out_f, "/gene_filter")
dir.create(out_f3)

sccount_df=read.table(low_exp_omit_path, header=1, row.names=1)

In [ ]:
intersect_genes=intersect(rownames(sccount_df), rownames(unique_df2))
length(intersect_genes)

##save Gene-TF binding site association matrix
intersect_TF_df=unique_df2[intersect_genes, ]
    

  
write.table(data.frame(gene=rownames(intersect_TF_df), intersect_TF_df), paste0(out_f3, "/TF_matrix.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)

##save RNA-seq logCPM
intersect_sc_df=logCPM[intersect_genes, ]


write.table(data.frame(gene=rownames(intersect_sc_df), intersect_sc_df), paste0(out_f3, "/transcriptome_matrix.txt"),row.names=FALSE,col.names=TRUE, sep="\t", append=F, quote=F)

